In [ ]:
import pandas as pd
import numpy as np


df = pd.read_csv("resultados.csv", sep=";")


SEP = "=" * 60


# 1. Substituir linhas vazias ou apenas com 'R$ ,' por NaN (nulo)
df['Preço'] = df['Preço'].replace(r'^\s*R\$\s*,*\s*$', np.nan, regex=True)
df['Preço'] = df['Preço'].replace('Sem preço', np.nan, regex=True)

# 2. Remover 'R$', converter vírgula decimal para ponto e tirar espaços
df['Preço'] = df['Preço'].str.replace('R$', '', regex=False)
df['Preço'] = df['Preço'].str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
df['Preço'] = df['Preço'].str.strip()

# 3. Tratar os valores nulos (NaN) preenchendo com 0, e converter para inteiro
df['Preço'] = df['Preço'].astype(float)
df['Preço'] = df['Preço'].fillna(0)

# 4. Remover as colunas com preços iguais a 0
df = df[df["Preço"] != 0]

In [ ]:
# Média de preço por loja para cada produto
for produto in df['Pesquisa'].unique():
    print(f"{SEP}\nProduto: {produto}\n{SEP}")
    medias = df[df['Pesquisa'] == produto].groupby('Loja')['Preço'].mean()
    print(round(medias, 2))
    
    loja_mais_barata = medias.idxmin()
    menor_preco      = medias.min()
    print(f"\n→ Site mais barato para '{produto}': {loja_mais_barata} (R$ {menor_preco:,.2f})")

# Site com menor preço no geral (média geral por loja)
print(f"\n{SEP}\nRANKING GERAL — MENOR PREÇO MÉDIO\n{SEP}")
media_geral = df.groupby('Loja')['Preço'].mean().sort_values()
print(media_geral)

vencedor       = media_geral.idxmin()
menor_media    = media_geral.min()
print(f"\n🏆 Site com menor preço no geral: {vencedor} (média R$ {menor_media:,.2f})")

In [ ]:
# ─────────────────────────────────────────────
# ESTATÍSTICAS COMPLETAS POR ITEM
# ─────────────────────────────────────────────
print(f"\n{SEP}\nESTATÍSTICAS POR ITEM\n{SEP}")

# Menor preço por item com a loja correspondente
idx_min = df.groupby('Pesquisa')['Preço'].idxmin()
idx_max = df.groupby('Pesquisa')['Preço'].idxmax()

stats = df.groupby('Pesquisa')['Preço'].agg(
    Menor_Preço='min',
    Maior_Preço='max',
    Preço_Médio='mean',
).reset_index()

# Adiciona a loja do menor e maior preço
stats['Loja_Menor'] = df.loc[idx_min, 'Loja'].values
stats['Loja_Maior'] = df.loc[idx_max, 'Loja'].values

# Variação percentual entre menor e maior
stats['Variação_%'] = (
    (stats['Maior_Preço'] - stats['Menor_Preço']) / stats['Menor_Preço'] * 100
).round(2)

# Economia por item
stats['Economia'] = stats['Maior_Preço'] - stats['Menor_Preço']

print(stats.to_string(index=False))

# ─────────────────────────────────────────────
# ECONOMIA TOTAL
# ─────────────────────────────────────────────
print(f"\n{SEP}\nECONOMIA TOTAL\n{SEP}")

menor_total  = stats['Menor_Preço'].sum()
maior_total  = stats['Maior_Preço'].sum()
economia     = maior_total - menor_total
economia_pct = economia / maior_total * 100

print(f"  Comprando sempre no menor preço : R$ {menor_total:>10,.2f}")
print(f"  Comprando sempre no maior preço : R$ {maior_total:>10,.2f}")
print(f"  Economia total                  : R$ {economia:>10,.2f}  ({economia_pct:.1f}%)")

# ─────────────────────────────────────────────
# DETALHES POR ITEM
# ─────────────────────────────────────────────
print(f"\n{SEP}\nDETALHES POR ITEM\n{SEP}")

for _, row in stats.iterrows():
    print(f"\n📦 {row['Pesquisa']}")
    print(f"   Menor preço : R$ {row['Menor_Preço']:,.2f}  ← {row['Loja_Menor']}")
    print(f"   Maior preço : R$ {row['Maior_Preço']:,.2f}  ← {row['Loja_Maior']}")
    print(f"   Preço médio : R$ {row['Preço_Médio']:,.2f}")
    print(f"   Variação    : {row['Variação_%']:.2f}%")
    print(f"   Economia    : R$ {row['Economia']:,.2f}")